# Präambel

In [1]:
from sympy import *
from IPython.display import display, Math, Latex, Markdown
#init_session()
init_printing(use_unicode=True)

In [2]:
class dotdict(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

def md(s):
    display(Markdown(s))
    
def ausnutzung(p, stab=None):
    color = '#000'
    emoji = ''
    if p > 1:
        color = '#f00'
        emoji = '🔴'
    elif p > 0.7:
        color = '#0f0'
        emoji = '🟢'
    elif p < 0.7:
        color = '#00f'
        emoji = '🟡'
    #md(f'#### <span style="color:{color};">{int(p*100)}%</span>')
    qs = stab.querschnitt if stab else ''
    md(f'#### {emoji} {int(N(p, 6)*100)}% ({qs})')
    return p


def note(s):
    md(f'<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">{s}</span></div>')
#ausnutzung(13.37)
#ausnutzung(0.238)
N(0.238, 2) * 100

In [3]:
%run querschnitte.ipynb

In [4]:
from dataclasses import dataclass
from decimal import Decimal
from functools import partial

@dataclass
class Schnittgroeszen:
    N: float = None
    V: float = 0
    Vz: float = 0
    My: float = 0
    Mz: float = 0
    ψ: float = None
    ψz: float = None
    
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    M = lambda self, achse: self.um_achse('M', achse)


    
@dataclass
class Stahlguete:
    guete: str
    E: int
    fy: float
    fu: float
    
@dataclass
class Stab(Querschnitt, Stahlguete):
    l: float
    sg: dict
    Lcry: float = None
    Lcrz: float = None
    

    def __init__(self, guete, QS, **kwargs):
        sg = kwargs.pop('sg', None)
        if isinstance(sg, dict):
            self.sg = sg
        else:
            self.sg = {None: sg}
            
        for k,v in kwargs.items():
            self.__setattr__(k, v)
            
        Querschnitt.__init__(self, **QS.__dict__)
        Stahlguete.__init__(self, **guete.__dict__)
        
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    Lcr = lambda self, achse: self.um_achse('Lcr', achse)

S235 = Stahlguete(guete='S235', E=21000, fy=23.5, fu=36)


# Nachweis-Funktionen

## Querschnitte

In [5]:
#import inspect
#inspect.getsourcelines(vglsp)


### Vergleichsspannung

In [182]:
def Sy(qs):
    Sf = (qs.b * qs.tf) * (qs.h/2 - qs.tf/2)
    Sw = ((qs.h/2 - qs.tf) * qs.tw) * (qs.h/2 - qs.tf)/2
    display(f'Sf = {Sf}   Sw = {Sw}')
    return Sf + Sw

def vglsp(stab, sg=None):
    last = stab.sg[sg]
    σx = last.N/stab.A
    if last.My:
        σx += (last.My*100)/stab.W('y')
    τ = -(last.V * Sy(stab))/(stab.I('y') * stab.tw) if last.V else 0
    display(f'sigmax = {N(σx,5)}  tau = {N(τ,5)}   S = {N(Sy(stab),5)} W = {N(stab.W('y'), 5)}')
    σv = sqrt(σx**2 + 3 * τ**2)
    return ausnutzung(N(σv/stab.fy, 5))

### Zug [EN 6.2.3]

In [7]:
def zug(stab, sg=None, A_net=None):
    γ_M0 = 1
    γ_M2 = 1.25
    N_plRd = (stab.A * stab.fy)/γ_M0
    N_uRd = (0.9 * A_net * stab.fu)/γ_M2 if A_net else oo
    last = stab.sg[sg]

    return ausnutzung(N(last.N/min(N_plRd, N_uRd), 3), stab)

### Druck [EN 6.2.4]

In [8]:
def druck(stab, sg=None):
    γ_M0 = 1
    N_cRd = (stab.A * stab.fy)/γ_M0
    last = stab.sg[sg]
    
    return ausnutzung(N(last.N/N_cRd, 3), stab)

### Biegung [EN 6.2.5]

In [9]:
def biegung(um_achse, stab, sg=None):
    γ_M0 = 1

    M_plRd = (stab.Wpl(um_achse) * stab.fy)/γ_M0
    
    #h = stab.h if um_achse == 'y' else stab.b
    #W_elmin = stab.I(um_achse)/(h/2)
    #M_elRd = (W_elmin * stab.fy)/γ_M0
    
    last = stab.sg[sg]
    return ausnutzung(N(last.M(um_achse)*100/M_plRd, 3), stab)

biegung_y = partial(biegung, 'y')
biegung_z = partial(biegung, 'z')

### Querkraft [EN 6.2.6] TODO: E-E

In [10]:
def querkraft(um_achse, stab, sg=None):
    γ_M0 = 1
    V_plRd = (stab.Av(um_achse) * (stab.fy/sqrt(3)) )/ γ_M0

    last = stab.sg[sg]
    return ausnutzung(last.V/V_plRd, stab)

querkraft_y = partial(querkraft, 'y')
querkraft_z = partial(querkraft, 'z')

### Kombi: M + N + V [EN 6.2.10]

In [11]:
def kombi_MNV(um_achse, stab, sg=None, debug=False):
    assert(um_achse == 'y'), 'Nur fuer My implementiert'
    γ_M0 = 1
    last = stab.sg[sg]
    
    V_plRd = (stab.Av(um_achse) * (stab.fy/sqrt(3)) )/ γ_M0
    assert((last.V / V_plRd) <= 0.5), 'Abminderung aufgrund V_Ed notwendig!'
    
    N_plRd = (stab.A * stab.fy)/γ_M0 
    M_NRd = M_plRd = (stab.Wpl(um_achse) * stab.fy)/γ_M0
    if um_achse == 'y' and ( (last.N > 0.25 * N_plRd) or (last.N > ((0.5 * stab.hw * stab.tw * stab.fy) / γ_M0)) ):
        n = last.N/N_plRd
        a = min(0.5, (stab.A - 2 * stab.b * stab.tf)/stab.A)
        M_NRd = min(M_plRd, M_plRd * (1-n)/(1-0.5*a))
        if debug == True:
            display(Math(f'n = {N(n, 5)}, a = {N(a, 5)}, M_NRd = {N(M_NRd, 5)}'))
    else:
        if debug == True:
            display(Math(f'M_NRd = {N(M_NRd, 5)}'))
    return ausnutzung(last.M(um_achse)*100/M_NRd, stab)

kombi_MNV_y = partial(kombi_MNV,'y')
kombi_MNV_z = partial(kombi_MNV,'z')

## Bauteil

In [191]:
γ_M1 = 1 #TODO tripple-check

def λ1(stab):
    return pi * sqrt(stab.E/stab.fy)

def λyz(um_achse, stab):
    return stab.Lcr(um_achse)/(stab.i(um_achse) * λ1(stab))

def KL2α(KL: str):
    return {'a0': 0.13, 'a': 0.21, 'b': 0.34, 'c': 0.49, 'd': 0.76}.get(KL)
    
    
def χyz(um_achse, stab, debug=False):
    λ = λyz(um_achse, stab)
    α = KL2α(stab.KL(um_achse))
    Φ = 0.5* (1 + α * (λ-0.2) + λ**2)
    chi = min(1, 1/(Φ + sqrt(Φ**2 - λ**2)))
    if debug:
        display(f'λ = {λ}  α = {α}  Φ = {Φ}  χ = {chi}')
    return chi

### Biegeknicken [B 8.7.6.1]

In [192]:
def biege_knicken(um_achse, stab, sg=None, debug=False):
    last = stab.sg[sg]
    assert(stab.Lcr(um_achse))
    assert(last.ψ is not None or last.My == 0)

    ## Widerstaende
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) # NB nur W_y relevant, egal um welche Achse
    My_Rd =  W_y * (stab.fy / γ_M1)

    ## Beiwerte
    n_y = last.N / (χyz('y', stab, debug=True) * N_Rd)
    k_y = min((1 + (λyz('y', stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    α = 0.6 if um_achse == 'z' else 1
    C_my = max(0.6 + 0.4 * (last.ψ or 0),
               0.4)
    if debug:
        display(Math(f'N_Rd = {N(N_Rd,5)}  My_Rd = {N(My_Rd,5)} \\, W_y = {N(W_y,3)}'))
        display(Math(f'n_y = {N(n_y,5)}  k_y = {N(k_y,5)} \\, C_my = {N(C_my,3)}'))
        display(Math(f'χ = {N(χyz(um_achse, stab), 5)}'))

    ## Nachweis
    res = (last.N/(χyz(um_achse, stab) * N_Rd)) + (α * k_y * (C_my * last.My)/My_Rd)
    return ausnutzung(N(res, 5), stab)

biege_knicken_y = partial(biege_knicken, 'y')
biege_knicken_z = partial(biege_knicken, 'z')

### Biegedrillknicken [B 8.7.6.1]

In [14]:
def χLT(stab): ## Gem. EC3 (6.3.2.3)
    λz = λyz('z', stab)
    
    k_c = 1 # Auf der sicheren Seite
    k_p = 0.9 * 1 / (1 + (1/20) * (λz/(stab.h/stab.tf))**2)**0.25
    λLT = k_c * k_p * λz
        
    λLT0 = 0.4  # Höchstwert (EC3 Seite 70)
    β = 0.75    # Mindestwert
    αLT = KL2α(stab.KL_BDK)
    
    ΦLT = 0.5 * ( 1 + αLT * (λLT - λLT0) + β * λLT**2)
    return min(1 / (ΦLT + sqrt(ΦLT**2 - β * λLT**2)),
               1,
               1 / (λLT**2),
              )

In [15]:
def BDK(um_achse, stab, sg=None):
    last = stab.sg[sg]
    assert(stab.Lcr(um_achse))
    assert(last.ψ is not None)
    
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) ## CHECK past?
    My_Rd =  W_y * (stab.fy / γ_M1) # W_y passt?

    
    ## Beiwerte
    C_my = max(0.4, (0.6 + 0.4 * last.ψ))
    C_MLT = C_my
    
    n_y = last.N / (χyz('y', stab) * N_Rd)
    k_y = min((1 + (λyz('y', stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    
    n_z = last.N / (χyz('z', stab) * N_Rd)
    λz = λyz('z', stab)
    assert (λz >= 0.4)
    k_LT = max(1 - (0.1 * n_z * λz)/(C_MLT - 0.25),
               1 - (0.1 * n_z     )/(C_MLT - 0.25),
              )


    ## vvv dragons
    #χ = χyz(um_achse, stab)

    interaktionsbeiwert = {'y': k_y * C_my,
                           'z': k_LT,
                          }.get(um_achse)

    res = (last.N/(χyz(um_achse, stab) * N_Rd)) + (interaktionsbeiwert * last.My/(χLT(stab) * My_Rd))
    return ausnutzung(N(res, 5), stab)

BDK_y = partial(BDK, 'y')
BDK_z = partial(BDK, 'z')

# Stäbe

In [16]:
staebe = dotdict()

## Pfette

In [179]:
staebe.pfette = Stab(
    S235, HEB180,
    l=675, Lcry=675, Lcrz=675,
    sg = {
        'druck': Schnittgroeszen(
            N  = 102,    
            My = 70.21, 
            V = 42.52, 
            ψ  = 0.95),
        'zug': Schnittgroeszen(
            N  = 69.1,   
            My = 70.21,
            V = 42.52,
            ψ  = 0.95),
        },
)
#done 2025-04-19 8:50

### Bauteil

In [193]:
biege_knicken_y(staebe.pfette, sg='druck', debug=True)
biege_knicken_z(staebe.pfette, sg='druck')

NameError: name 'χ' is not defined

### Querschnitt

In [183]:
#zug(staebe.pfette, sg='zug')
#kombi_MNV_y(staebe.pfette, sg='zug')
#kombi_MNV_y(staebe.pfette, sg='druck')
#querkraft_y(staebe.pfette, sg='druck')

In [184]:
vglsp(staebe.pfette, sg='druck')
vglsp(staebe.pfette, sg='zug')

'Sf = 209.16000000000003   Sw = 24.548'

'Sf = 209.16000000000003   Sw = 24.548'

'sigmax = 18.057  tau = -3.0517   S = 233.71 W = 425.67'

#### 🟢 80% ()

'Sf = 209.16000000000003   Sw = 24.548'

'Sf = 209.16000000000003   Sw = 24.548'

'sigmax = 17.553  tau = -3.0517   S = 233.71 W = 425.67'

#### 🟢 78% ()

## OG

In [163]:
staebe.obergurt = Stab(
    S235, HEB320,
    l=250.3, Lcry=250.3, Lcrz=250.3,
    sg = {
        'zug': Schnittgroeszen(
            N  = 2090.88,
            My = 118.01,
            Mz = 30.51,
            V  = 44.59,
            Vz = 13.29,
            ψ  = 28.74/118.01,
            ψz = 9.71/25.38,
        ),
        'druck': Schnittgroeszen(
            N  = 317.92,
            #My = 14.69,
            My = 118.01,
            Mz = 15.08,
            #V  = 7.01,
            V  = 44.59,
            Vz = 6.44,
            ψz = 3.84/14.85, # anderer verlauf, aber maximum
            ψ  = 4.07/14.69,
        ),
    })
#done 2025-04-19 8:50

### Querschnitt

In [164]:
zug(staebe.obergurt, sg='zug')
kombi_MNV_y(staebe.obergurt, sg='zug')
#kombi_MNV_z(staebe.obergurt, sg='zug')
#TODO zug um z!! (?) 

kombi_MNV_y(staebe.obergurt, sg='druck')

#### 🟡 55% (HEB 320)

#### 🟡 45% (HEB 320)

#### 🟡 23% (HEB 320)

In [165]:
vglsp(staebe.obergurt, sg='zug')

'sigmax = 19.089  tau = -1.2994   S = 1032.9 W = 1926.3'

#### 🟢 81% ()

### Bauteil

In [167]:
biege_knicken_y(staebe.obergurt, sg='druck')
biege_knicken_z(staebe.obergurt, sg='druck')

#### 🟡 8% (HEB 320)

#### 🟡 9% (HEB 320)

## UG Feld

In [168]:
staebe.untergurt_feld = Stab(
    S235, HEB320,
    l=0, Lcry=240, Lcrz=378.1,
    sg = {
        'druck': Schnittgroeszen(
            N  = 2532.81,
            My = 58.98,
            V  = 42.88,
            ψ  = 1
        ),
        'zug': Schnittgroeszen(
            N  = 384.45,
            #My = 8.85,
            My = 58.98,
            #V  = 7.21,
            V  = 42.88,
            ψ  = 1,
        ),
    })
#done 2025-04-19 8:56

### Querschnitt

In [169]:
#kombi_MNV_y(staebe.untergurt_feld, sg='zug-y')
#kombi_MNV_y(staebe.untergurt_feld, sg='druck-y')
#druck(staebe.untergurt_feld, sg='druck-y')

In [170]:
vglsp(staebe.untergurt_feld, sg='zug')
vglsp(staebe.untergurt_feld, sg='druck')

'sigmax = 5.4454  tau = -1.2496   S = 1032.9 W = 1926.3'

#### 🟡 24% ()

'sigmax = 18.764  tau = -1.2496   S = 1032.9 W = 1926.3'

#### 🟢 80% ()

### Bauteil

In [171]:
BDK_y(staebe.untergurt_feld, sg='druck')

#### 🟡 66% (HEB 320)

In [172]:
BDK_z(staebe.untergurt_feld, sg='druck')

#### 🟢 81% (HEB 320)

## UG Kragarm

In [174]:
staebe.untergurt_krag_li = Stab(
    S235, HEB300,
    l=256, Lcry=256, Lcrz=648.7,
    #l=256, Lcry=256, Lcrz=3.95,
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 1867.9,
            My = 34.38,
            V  = 25.95,
            ψ  = 27/34,
        ),
        'druck-z': Schnittgroeszen(
            N  = 1867.9,
            My = 34.38,
            V  = 25.95,
            ψ  = 0,
        ),
        'zug-y': Schnittgroeszen(
            N  = 188.58,
            #My = 3.63,
            #V  = 3.04,
            My = 34.38,
            V  = 25.95,
            ψ  = 2.2/3.1, 
        ),
        'zug-z': Schnittgroeszen(
            N  = 188.58,
            My = 34.38,
            V  = 25.95,
            #My = 3.63,
            #V  = 3.04,
            ψ  = 0,
        ),
    })
# ^^^ done 2025-04-19 8:06

staebe.untergurt_krag_re = Stab(
    S235, HEB300,
    l=0, Lcry=253, Lcrz=2*485.8,
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 453.44,
            My = 14.61,
            V  = 6.45,
            ψ  = 3.32/14.64,
        ),
        'druck-z': Schnittgroeszen(
            N  = 453.44,
            My = 14.61,
            V  = 6.45,
            ψ  = 0
        ),
    })
# ^^^ done 2025-04-19 09:11

note('zug UG krag re nicht relevant, weil schon mit UG krag li abgedeckt')
note('ψ werte fuer -z stab doublechecken!!')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">zug UG krag re nicht relevant, weil schon mit UG krag li abgedeckt</span></div>

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">ψ werte fuer -z stab doublechecken!!</span></div>

### Querschnitt

In [175]:
#zug(staebe.untergurt_krag_li, sg='zug-y')
#kombi_MNV_y(staebe.untergurt_krag_li, sg='zug-y')
#kombi_MNV_y(staebe.untergurt_krag_li, sg='druck-y')
#kombi_MNV_y(staebe.untergurt_krag_re, sg='druck-y')

In [176]:
vglsp(staebe.untergurt_krag_li, sg='zug-y')
vglsp(staebe.untergurt_krag_li, sg='druck-y')
vglsp(staebe.untergurt_krag_re, sg='druck-y')

'sigmax = 3.3137  tau = -0.83907   S = 895.24 W = 1678.0'

#### 🟡 15% ()

'sigmax = 14.577  tau = -0.83907   S = 895.24 W = 1678.0'

#### 🟡 62% ()

'sigmax = 3.9119  tau = -0.20856   S = 895.24 W = 1678.0'

#### 🟡 16% ()

### Bauteil

In [177]:
BDK_y(staebe.untergurt_krag_li, sg='druck-y')
BDK_z(staebe.untergurt_krag_li, sg='druck-z')


#### 🟡 53% (HEB 300)

#### 🟢 89% (HEB 300)

In [178]:
BDK_y(staebe.untergurt_krag_re, sg='druck-y')
BDK_z(staebe.untergurt_krag_re, sg='druck-z')

#### 🟡 13% (HEB 300)

#### 🟡 35% (HEB 300)

## Streben FW

In [88]:
staebe.strebe = Stab(
    S235, HEA160,
    l=192, Lcry=192, Lcrz=192,
    sg = {
        'druck': Schnittgroeszen(
            N  = 632.36,
            My = 0,
            V  = 0,
        ),
        'zug': Schnittgroeszen(
            N  = 766.91,
            My = 0,
            V  = 0,
        ),
    }
)
# ^^^ done 2025-04-19 09:17


### Querschnitt (Zug)

In [89]:
#zug(staebe.strebe, sg='zug')
vglsp(staebe.strebe, sg='zug')

'sigmax = 19.781  tau = 0   S = 116.43 W = 220.13'

#### 🟢 84% ()

### Bauteil

In [90]:
biege_knicken_y(staebe.strebe, sg='druck')
biege_knicken_z(staebe.strebe, sg='druck')

#### 🟢 72% (HEA 160)

#### 🟢 83% (HEA 160)

## Streben UG

In [91]:
staebe.strebe_UG = Stab(
    S235, HEB180, l=149, Lcry=149, Lcrz=149,
    sg = {
        'druck': Schnittgroeszen(
            N  = 1233.37,
            My = 0,
            V  = 0,
        ),
        'zug': Schnittgroeszen(
            N  = 199.97, 
            My = 0,
            V  = 0,
        ),
    }
)
# ^^^ done 2025-04-19 9:24

### Querschnitt

In [94]:
#zug(staebe.strebe_UG, sg='zug')
#druck(staebe.strebe_UG, sg='druck')
vglsp(staebe.strebe_UG, sg='zug')
vglsp(staebe.strebe_UG, sg='druck')

'sigmax = 3.0647  tau = 0   S = 233.71 W = 425.67'

#### 🟡 13% ()

'sigmax = 18.902  tau = 0   S = 233.71 W = 425.67'

#### 🟢 80% ()

### Bauteil

In [95]:
biege_knicken_y(staebe.strebe_UG, sg='druck')
biege_knicken_z(staebe.strebe_UG, sg='druck')

#### 🟢 80% (HEB 180)

#### 🟢 86% (HEB 180)

## Knickpfette

In [97]:
staebe.knickpfette = Stab(
    S235, IPE160, l=675, Lcry=675, Lcrz=675,
    sg = {
        'druck': Schnittgroeszen(
            N  = 25.07,
            My = 1.3,
            V  = 0.8,
            ψ  = 0.95,
        ),
        #'zug': Schnittgroeszen(
        #    N  = 0 
        #    My = 0,
        #    V  = 0,
        #),
    }
)
# ^^^ done 2025-04-19 9:28

In [98]:
druck(staebe.knickpfette, sg='druck')
kombi_MNV_y(staebe.knickpfette, sg='druck')
vglsp(staebe.knickpfette, sg='druck')

BDK_y(staebe.knickpfette, sg='druck')
BDK_z(staebe.knickpfette, sg='druck')
biege_knicken_y(staebe.knickpfette, sg='druck')
biege_knicken_z(staebe.knickpfette, sg='druck')

#### 🟡 5% (IPE 160)

#### 🟡 4% (IPE 160)

'sigmax = 2.3208  tau = -0.098175   S = 59.476 W = 121.16'

#### 🟡 9% ()

#### 🟡 9% (IPE 160)

#### 🟢 88% (IPE 160)

#### 🟡 8% (IPE 160)

#### 🟢 88% (IPE 160)

## Dachverband

In [99]:
staebe.dachverband = Stab(
    S235, RD30, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 150.48, 
        My = 0,
        V  = 0,
    ),
)
# ^^^ done 2025-04-19 9:33

### Querschnitt

In [101]:
vglsp(staebe.dachverband)

'sigmax = 21.284  tau = 0   S = 0 W = 0'

#### 🟢 90% ()

## 6 Wandverband

In [109]:
staebe.wandverband = Stab(
    S235, RD45, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 354.59,
        My = 0,
        V  = 0,
    ),
)

### Querschnitt

In [110]:
#zug(staebe.wandverband)
vglsp(staebe.wandverband)

'sigmax = 22.301  tau = 0   S = 0 W = 0'

#### 🟢 94% ()

## Knickverband

In [111]:
staebe.knickverband = Stab(
    S235, RD20, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 64.24,
        My = 0,
        V  = 0,
    ),
)
# ^^^ done 2025-04-19 9:39


### Querschnitt

In [113]:
vglsp(staebe.knickverband)

'sigmax = 20.459  tau = 0   S = 0 W = 0'

#### 🟢 87% ()

## 7 Stütze A

In [132]:
staebe.stuetzeA = Stab(
    S235, HEA160, l=650, Lcry=650, Lcrz=650,
    sg = {
        'zug': Schnittgroeszen(
            N  = 692.89,# + (-141.98),
            My = 0,
            V  = 0,
        ),
        'druck': Schnittgroeszen(
            N  = 113.45 + (97.76),
            My = 0,
            V  = 0,
        ),
    })

# ^^^ done 2025-04-19 9:47
note('dachverband einfluss weggelassen, auf sicheren seite (eh nicht maßgebend)')
note('HEA140 waren 93%, aber keine schraubenloecher geplant ansich... fuer spaetere adaptionen HEA160?')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">dachverband einfluss weggelassen, auf sicheren seite (eh nicht maßgebend)</span></div>

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">HEA140 waren 93%, aber keine schraubenloecher geplant ansich... fuer spaetere adaptionen HEA160?</span></div>

### Querschnitt

In [130]:
zug(staebe.stuetzeA, sg='zug')
druck(staebe.stuetzeA, sg='druck')
vglsp(staebe.stuetzeA, sg='zug')
vglsp(staebe.stuetzeA, sg='druck')

#### 🟢 76% (HEA 160)

#### 🟡 23% (HEA 160)

'sigmax = 17.872  tau = 0   S = 116.43 W = 220.13'

#### 🟢 76% ()

'sigmax = 5.4478  tau = 0   S = 116.43 W = 220.13'

#### 🟡 23% ()

### Bauteil

In [131]:
biege_knicken_y(staebe.stuetzeA, sg='druck')
biege_knicken_z(staebe.stuetzeA, sg='druck')

#### 🟡 41% (HEA 160)

#### 🟢 93% (HEA 160)

## 8 Stütze B

In [156]:
staebe.stuetzeB = Stab(
    S235, HEB700, l=775, Lcry=0, Lcrz=625,
    sg = {
        'druck-max': Schnittgroeszen(
            N  = 1332.1,
            My = 1044.13,
            V  = 696.43,
            ψ  = 1,
        ),
        'druck-maxM': Schnittgroeszen(
            N  = 1332.1,
            My = 1044.13,
            V  = 696.43,
            ψ  = 1,
        ),
        'druck-stabenden': Schnittgroeszen(
            N  = 1332.1,
            My = 1044.13,
            V  = 696.43,
            ψ  = 1,
        ),
        'zug': Schnittgroeszen(
            N  = 167.46,
            My = 235.99,
            V  = 157.33,
        ),
    })

### Knicklänge (um y)

In [157]:
h_Fachwerk = 108                 # Achsabstand der Gurte in Mitte des Feldbereichs vom Fachwerk
z = h_Fachwerk * 0.5
l_R = 500
l = 775
l1 = staebe.stuetzeA.l

N_A = abs(-staebe.stuetzeA.sg['zug'].N)
N_B = abs(staebe.stuetzeB.sg['druck-max'].N)
F = N_A * (l / l1)   # <--- Einsparungspotential!  ggf hoehe bis traegermitte waehlbar

display(f'NA = {N_A}')
display(f'NB = {N_B}')
note('auf sicheren Seite mit absolutwerten gerechnet')

EI = staebe.stuetzeB.E * staebe.stuetzeB.Iy
EI_R = staebe.obergurt.E * ((staebe.obergurt.Iy + staebe.obergurt.A * z**2) + (staebe.untergurt_feld.Iy + staebe.untergurt_feld.A * z**2))

ν = 1 / ( (l_R * EI)/(l * EI_R) + 1)
η = ( (3 * ν) / ( (0.216 * ν**2 + 1) * N_B + F)) * (EI/l**2)
staebe.stuetzeB.Lcry = N(pi * sqrt(EI/(η*N_B)))
staebe.stuetzeB.Lcry

'NA = 692.89'

'NB = 1332.1'

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">auf sicheren Seite mit absolutwerten gerechnet</span></div>

### Querschnitt

In [158]:
#zug(staebe.stuetzeB, sg='zug')
#druck(staebe.stuetzeB, sg='druck-y')
#querkraft_y(staebe.stuetzeB, sg='druck-y')
#kombi_MNV_y(staebe.stuetzeB, sg='zug')
#kombi_MNV_y(staebe.stuetzeB, sg='druck-y')
vglsp(staebe.stuetzeB, sg='zug')
vglsp(staebe.stuetzeB, sg='druck-max')

'sigmax = 3.7617  tau = -1.4647   S = 4066.0 W = 7340.0'

#### 🟡 19% ()

'sigmax = 18.573  tau = -6.4838   S = 4066.0 W = 7340.0'

#### 🟢 92% ()

### Bauteil

In [159]:
BDK_y(staebe.stuetzeB, sg='druck-max')

#### 🟡 23% (HEB 700)

In [160]:
biege_knicken_z(staebe.stuetzeB, sg='druck-max')

#### 🟡 30% (HEB 700)